# Notebook 13 — Regex & text hygiene

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `12-logging-for-pipelines.ipynb`

**Next up:** `14-bytes-encoding-files.ipynb`

---

## Learning objectives

- Compile patterns once when reused.
- Prefer parsers over regex for HTML — but regex shines at fence/tag hygiene.
- Know greedy vs non-greedy boundaries around citations.

## Table of contents

1. **Compiled patterns**
2. **Citation-shaped extraction**
3. **Fence stripping mindset**
4. **Progressive drills — digits → bracket tags → fenced JSON blocks**
5. **Exercise — strip loose citations**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Compiled patterns

*Explanation:* **`re.compile`** amortizes parsing cost across thousands of chunks.


In [ ]:
import re

MODEL_REF = re.compile(r"model[=:]\s*(\w+)", re.I)

text = "Retry model=gpt-mini then Model: opus"
print(MODEL_REF.findall(text))


## 2 · Citation-shaped extraction

*Explanation:* Retrieval QA answers often emit `[source: doc]` snippets — normalize before persistence.


In [ ]:
import re

CITE = re.compile(r"\[source:\s*([^\]]+)\]", re.I)

answer = "Latency dropped [source: runbook section 4] after caching."
print(CITE.findall(answer))


## 3 · Fence stripping mindset

*Explanation:* Models wrap JSON inside markdown fences — peel fences before `json.loads`.


In [ ]:
import json
import re

FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.S | re.I)


def loads_maybe_fenced(blob: str) -> dict:
    m = FENCE.search(blob)
    core = m.group(1) if m else blob
    return json.loads(core.strip())


fence_json = "```json"
body = '{"tool": "search", "query": "logs"}'
raw = "Here you go:\n" + fence_json + "\n" + body + "\n" + "```" + "\n"
print(loads_maybe_fenced(raw))


---

## Progressive drills — **easy → harder**

**Messy strings** arrive hourly — drills tighten hygiene without brittle copy/paste parsers.


### A · Easiest — grab integers

Pull numeric budgets out of prose instructions.


In [ ]:
import re

budget = re.findall(r"max_tokens[=:]\s*(\d+)", "please cap max_tokens: 2048 please")
print(budget)


### B · Medium — normalize whitespace collapse

Collapse accidental double spaces before hashing prompts.


In [ ]:
import re

messy = "hello   world\n\tagain"
clean = re.sub(r"\s+", " ", messy.strip())
print(clean)


### C · Harder — capture balanced-ish bullet IDs

Mirror extracting `[chunk::uuid]` markers.


In [ ]:
import re

tag = re.compile(r"\[chunk::([a-z0-9\-]{8,})\]", re.I)
text = "See [chunk::ab12cd34-eeee] for detail."
print(tag.findall(text))


### Exercise — citations only

Implement `strip_citations(answer: str) -> str` removing every `[source: ...]` substring (including brackets), leaving single spaces where multiples collapsed.


In [ ]:
import re


def strip_citations(answer: str) -> str:
    raise NotImplementedError


dirty = "OK [source: a] maybe [SOURCE: second]"
assert strip_citations(dirty).strip() == "OK maybe"
print("OK")


### Solution — strip_citations

<details>
<summary>Click to expand</summary>

```python
import re

_CITES = re.compile(r"\[source:\s*[^\]]+\]", re.I)

def strip_citations(answer: str) -> str:
    return re.sub(r"\s+", " ", _CITES.sub("", answer)).strip()
```

</details>
